In [5]:
from pathlib import Path
import shutil


def infer_base_label(folder_name: str) -> str | None:
    """
    Examples
    --------
    241213 first-pattern -> first_pattern
    241214 pattern       -> pattern
    241224 position      -> position
    """
    name = folder_name.lower()

    has_first = "first" in name
    has_pattern = ("pattern" in name) or ("pat" in name)
    has_position = ("position" in name) or ("pos" in name)

    if has_first and has_pattern:
        return "first-pattern"
    if has_first and has_position:
        return "first-position"
    if (not has_first) and has_pattern:
        return "pattern"
    if (not has_first) and has_position:
        return "position"

    return None


def infer_cutoff_label(folder_name: str) -> str | None:
    name = folder_name.lower().replace("_", " ")

    if "cut off 0" in name:
        return "cut-off-0"

    if "cut off top250" in name:
        return "cut-off-top250"

    return None


def collect_neuro_type_mat_files(
    source_root,
    data_root,
    overwrite: bool = False,
    verbose: bool = True,
):
    """
    从原始 2P 数据结构中收集 neuro_type 开头的 .mat 文件。

    Parameters
    ----------
    source_root : str or Path
        原始数据根目录，例如 "Backup1/LWX/Processed 2P"。

    data_root : str or Path
        输出目录，例如 "./data"。

    overwrite : bool
        如果目标文件已存在，是否覆盖。

    verbose : bool
        是否打印复制过程。

    Returns
    -------
    copied_files : list[tuple[Path, Path]]
        每个元素是 (源文件路径, 目标文件路径)。
    """
    source_root = Path(source_root)
    data_root = Path(data_root)

    if not source_root.exists():
        raise FileNotFoundError(f"source_root does not exist: {source_root}")

    data_root.mkdir(parents=True, exist_ok=True)

    copied_files = []

    # 最外层：mouse 文件夹，例如 HP01, HP02, HPC13
    for mouse_dir in source_root.iterdir():
        if not mouse_dir.is_dir():
            continue

        mouse_name = mouse_dir.name
        out_mouse_dir = data_root / mouse_name
        out_mouse_dir.mkdir(parents=True, exist_ok=True)

        # 第二层：实验目录，例如 241213 first-pattern
        for exp_dir in mouse_dir.iterdir():
            if not exp_dir.is_dir():
                continue

            base_label = infer_base_label(exp_dir.name)
            if base_label is None:
                if verbose:
                    print(f"[Skip exp] Cannot infer label: {exp_dir}")
                continue

            # 第三层：只查找 cut off 0 和 cut off top250
            for cutoff_dir in exp_dir.iterdir():
                if not cutoff_dir.is_dir():
                    continue

                cutoff_label = infer_cutoff_label(cutoff_dir.name)
                if cutoff_label is None:
                    continue

                full_label = f"{base_label}_{cutoff_label}"

                # 找 neuro_type 开头的 mat 文件
                mat_files = sorted(cutoff_dir.glob("neuro_type*.mat"))

                if len(mat_files) == 0:
                    if verbose:
                        print(f"[No mat] {cutoff_dir}")
                    continue

                for mat_file in mat_files:
                    new_name = f"{mat_file.stem}_{full_label}{mat_file.suffix}"
                    dst_file = out_mouse_dir / new_name

                    if dst_file.exists() and not overwrite:
                        if verbose:
                            print(f"[Exists, skip] {dst_file}")
                        continue

                    shutil.copy2(mat_file, dst_file)
                    copied_files.append((mat_file, dst_file))

                    if verbose:
                        print(f"[Copied] {mat_file} -> {dst_file}")

    return copied_files

In [6]:
source_root = r"/Backup1/LWX/Processed 2P/"
data_root = r"../../data/HPC_2p"

copied = collect_neuro_type_mat_files(
    source_root=source_root,
    data_root=data_root,
    overwrite=False,
    verbose=True,
)

print(f"Copied {len(copied)} files.")

[Exists, skip] ../../data/HPC_2p/HP24/neuro_type_saveHP24_25_2025-08-08_position_cut-off-0.mat
[Exists, skip] ../../data/HPC_2p/HP24/neuro_type_saveHP24_25_2025-08-08_position_cut-off-top250.mat
[Exists, skip] ../../data/HPC_2p/HP24/neuro_type_saveHP24_19_2025-08-01_pattern_cut-off-0.mat
[Exists, skip] ../../data/HPC_2p/HP24/neuro_type_saveHP24_19_2025-08-01_pattern_cut-off-top250.mat
[Exists, skip] ../../data/HPC_2p/HP24/neuro_type_saveHP24_18-2_2025-07-30_first-pattern_cut-off-0.mat
[Exists, skip] ../../data/HPC_2p/HP24/neuro_type_saveHP24_18-2_2025-07-30_first-pattern_cut-off-top250.mat
[Exists, skip] ../../data/HPC_2p/HP29/neuro_type_saveHP29_18_2025-08-13_pattern_cut-off-0.mat
[Exists, skip] ../../data/HPC_2p/HP29/neuro_type_saveHP29_18_2025-08-13_pattern_cut-off-top250.mat
[Exists, skip] ../../data/HPC_2p/HP29/neuro_type_saveHP29_10-2_2025-08-04_first-position_cut-off-0.mat
[Exists, skip] ../../data/HPC_2p/HP29/neuro_type_saveHP29_10-2_2025-08-04_first-position_cut-off-top250.mat